# 공정 비교: 8bin·9bin의 세 가지 방식 vs 분위수 thermometer·공식 XFeat

새 CPU Colab 런타임에서 이 파일 하나만 **모두 실행**하세요. AVX2 x86 CPU가 필요합니다.

각 B=8,9에 아래 방법을 실행합니다. 원본 XFeat 키포인트/디스크립터는 모든 방법에 동일합니다.

| 항목 | Projection | 경계 | 저장 코드 | 거리 |
|---|---|---|---|---|
| 기준: 기존 분위수 thermometer | 없음 | 분위수 | B−1개의 threshold bit | XOR/popcount = bin ABS |
| 1. k-means thermometer | 없음 | 차원별 k-means(B) 중심 사이 중간값 | B−1개의 threshold bit | XOR/popcount = bin ABS |
| 2. Manhattan PCA-MQ | PCA 64→64 | PCA 출력에 차원별 k-means(B) | 일반 이진 bin 인덱스 NBC | 숫자 인덱스 ABS |
| 3. 분위수 NBC ABS | 없음 | 기준 thermometer와 **완전히 같은** 분위수 경계 | 일반 이진 bin 인덱스 NBC | 숫자 인덱스 ABS |

- 공식 XFeat + 기준 2개 + 요청 6개 = **9개 방법**. 10·11bin 및 PCA+k-means thermometer는 제외했습니다.
- Manhattan 원문: https://cs.nju.edu.cn/lwj/paper/SIGIR12_MH.pdf (§3.2–3.3). PCA-MQ는 논문에 명시된 projection 선택입니다.
- **8bin은 q=3인 논문 방식. 9bin은 확장 실험:** 원 논문은 2^q개의 bin을 사용하므로 q=4라면 원래 16bin입니다. 여기서는 요청에 따라 k=9를 쓰고 인덱스 0..8을 4bit로 저장합니다. 9bin을 원 논문의 정확한 재현으로 표시하지 않습니다.
- 8bin NBC: 3bit×64=24B/KP; 9bin NBC: 4bit×64=32B/KP. Thermometer: 각각 56B/64B. NBC는 일반 이진 코드에 Hamming을 적용하지 않습니다.
- NBC 코드는 매칭 호출 안에서 uint8로 풀어 AVX2 SAD로 정확한 ABS를 계산합니다. **unpack 시간도 포함**, scratch 64B/KP는 저장 크기와 구분해 기록합니다.
- PCA·k-means·분위수 경계는 calibration 데이터로만 한 번 계산합니다. Synthetic held-out·HPatches는 학습에 쓰지 않습니다.
- 주 시간 `total_ms`는 projection(사용 시), 양자화, packing/layout, unpack 및 양방향 MNN을 포함하고 특징 추출은 제외합니다. 학습 시간은 별도 기록합니다.
- `cached_ms`는 미리 인코딩된 데이터의 매칭만 측정합니다. `pipeline_ms`는 공식 특징 추출까지 포함합니다.
- 같은 CPU 2스레드로 27회 매칭·18회 pipeline 측정. 9회마다 각 방법의 실행 위치가 균등하게 배정됩니다.
- 3번과 기준 thermometer는 같은 bin 번호를 쓰므로 모든 거리와 MNN이 같아야 합니다. 모든 평가 쌍에서 이를 검증합니다.
- Precision@3px, 정답/매칭 수, 중앙 오차, 저장 크기, 속도, 공식 대비 가속을 표에 표시합니다. P10/P90은 반복 분포이며 신뢰구간이 아닙니다.
- `results/fair_comparison_8_9bin_three_methods/실행시각/`에 CSV, JSON, 원시 시간, 모든 경계, PCA 및 k-means 중심을 저장합니다.


In [1]:
!pip -q install opencv-python-headless numba pandas scikit-learn threadpoolctl

import os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
import time
import io
import zipfile
import subprocess
import shutil
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from numba import njit, prange, set_num_threads, get_num_threads
from google.colab import files, drive

SEED = 7
TOP_K = 800
LEVELS = 8
CALIBRATION_WARPS_PER_IMAGE = 6
VALIDATION_WARPS_PER_IMAGE = 3
MAX_POSITIVES_PER_PAIR = 500
GEOMETRIC_THRESHOLD = 3.0
NUMBA_THREADS = 2
RUN_HPatches = True
TIMING_PAIRS_PER_DATASET = 5  # evenly spaced pairs; compare every method on each pair
WARMUP = 3
REPEATS = 27  # 9 methods, 3 balanced blocks

set_num_threads(NUMBA_THREADS)
torch.set_num_threads(NUMBA_THREADS)
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = 'cpu'  # identical CPU execution for official and proposed pipelines

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    if torch.get_num_interop_threads() != 1:
        raise RuntimeError('Restart the runtime before running this comparison.')
xfeat = torch.hub.load('verlab/accelerated_features', 'XFeat', pretrained=True, top_k=TOP_K).to(device).eval()
xfeat.dev = torch.device(device)  # official preprocessing follows this device
for parameter in xfeat.parameters():
    parameter.requires_grad_(False)
print('Device:', device)
PIPELINE_REPEATS = 18  # 9 methods, 2 balanced blocks
TIMING_SEED = 20261008
assert torch.get_num_threads() == NUMBA_THREADS == 2
# COCO is used for calibration and an internal synthetic-warp check.
# MH controls learn PCA/k-means using calibration only. HPatches is reserved for the external real-photo evaluation at the end.
DATA_SOURCE = 'coco'  # change to 'upload' to use your own JPG/PNG files
COCO_SOURCE_COUNT = 24
COCO_URL = 'https://images.cocodataset.org/zips/val2017.zip'
drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/Colab Notebooks/CSE527_datasets')
data_dir.mkdir(parents=True, exist_ok=True)
RESULT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/success-direct-thermometer-AVX2-SIMD4/results')
print('Persistent dataset cache:', data_dir)

def download_archive(url, destination):
    if destination.exists() and zipfile.is_zipfile(destination):
        print('Using persistent Drive cache:', destination)
        return
    # Reuse a download from an earlier run in this same Colab session, if present.
    legacy = Path('/content/xfeat_data') / destination.name
    if legacy.exists() and zipfile.is_zipfile(legacy):
        shutil.copy2(legacy, destination)
        print('Copied previous runtime download into Drive:', destination)
        return
    print('Downloading once into Drive:', url)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', str(destination), url], check=True)
    if not zipfile.is_zipfile(destination):
        raise RuntimeError(f'Incomplete or invalid ZIP: {destination}. Delete it and rerun this cell.')

if DATA_SOURCE == 'coco':
    coco_zip = data_dir / 'val2017.zip'
    download_archive(COCO_URL, coco_zip)
    with zipfile.ZipFile(coco_zip) as archive:
        names = sorted(n for n in archive.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png')))
        if len(names) < COCO_SOURCE_COUNT:
            raise RuntimeError('COCO archive has fewer images than requested.')
        selected = np.random.default_rng(SEED).choice(len(names), COCO_SOURCE_COUNT, replace=False)
        image_items = [(Path(names[i]).name, archive.read(names[i])) for i in selected]
    print(f'Selected {len(image_items)} COCO images using seed {SEED}; no COCO annotations needed.')
elif DATA_SOURCE == 'upload':
    print('Upload JPG/PNG images from distinct scenes.')
    uploaded = files.upload()
    image_items = list(uploaded.items())
    if not image_items:
        raise ValueError('Upload at least one image.')
else:
    raise ValueError("DATA_SOURCE must be 'coco' or 'upload'")

def decode_and_resize(raw_bytes, maximum_side=640):
    image = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError('An uploaded file is not a readable JPG/PNG image.')
    height, width = image.shape[:2]
    scale = min(1.0, maximum_side / max(height, width))
    new_width = max(64, (int(width * scale) // 32) * 32)
    new_height = max(64, (int(height * scale) // 32) * 32)
    return cv2.resize(image, (new_width, new_height))

# Split by source image, before any synthetic warps or descriptor calibration.
# Upload images of distinct scenes; near-duplicate photos belong to the same split.
image_items = sorted(image_items)
images = [decode_and_resize(raw) for _, raw in image_items]
if len(images) >= 2:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(images))
    validation_count = max(1, int(np.ceil(0.25 * len(images))))
    validation_ids = set(order[:validation_count].tolist())
    calibration_images = [im for i, im in enumerate(images) if i not in validation_ids]
    validation_images = [im for i, im in enumerate(images) if i in validation_ids]
    print('Calibration files:', [name for i, (name, _) in enumerate(image_items) if i not in validation_ids])
    print('Held-out files:', [name for i, (name, _) in enumerate(image_items) if i in validation_ids])
else:
    calibration_images = validation_images = images
    print('WARNING: One source image only; validation shares scene content with training.')
print('Calibration images:', len(calibration_images), 'Validation images:', len(validation_images))

The repository verlab_accelerated_features does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/verlab/accelerated_features/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/verlab/accelerated_features/raw/main/weights/xfeat.pt" to /root/.cache/torch/hub/checkpoints/xfeat.pt


100%|██████████| 5.96M/5.96M [00:01<00:00, 6.07MB/s]


Device: cpu
Mounted at /content/drive
Persistent dataset cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets
Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/val2017.zip
Selected 24 COCO images using seed 7; no COCO annotations needed.
Calibration files: ['000000002261.jpg', '000000029675.jpg', '000000067534.jpg', '000000162858.jpg', '000000165713.jpg', '000000173008.jpg', '000000175364.jpg', '000000194506.jpg', '000000270402.jpg', '000000289343.jpg', '000000360097.jpg', '000000460841.jpg', '000000471869.jpg', '000000479596.jpg', '000000504711.jpg', '000000517832.jpg', '000000528524.jpg', '000000547383.jpg']
Held-out files: ['000000077595.jpg', '000000132544.jpg', '000000334006.jpg', '000000394559.jpg', '000000449432.jpg', '000000474039.jpg']
Calibration images: 18 Validation images: 6


## 동일 특징 추출과 calibration / validation 분리

In [2]:
def image_to_tensor(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()[None].to(device) / 255.0

@torch.inference_mode()
def extract_features(image):
    output = xfeat.detectAndCompute(image_to_tensor(image), top_k=TOP_K)[0]
    return {
        'keypoints': output['keypoints'].detach().cpu().float(),
        'descriptors': output['descriptors'].detach().cpu().float(),
    }

def make_random_homography(height, width, generator):
    center = (width / 2.0, height / 2.0)
    angle = generator.uniform(-20.0, 20.0)
    scale = generator.uniform(0.88, 1.08)
    affine = cv2.getRotationMatrix2D(center, angle, scale)
    affine[0, 2] += generator.uniform(-0.06, 0.06) * width
    affine[1, 2] += generator.uniform(-0.06, 0.06) * height
    homography = np.vstack([affine, [0., 0., 1.]]).astype(np.float32)
    homography[2, 0] = generator.uniform(-7e-5, 7e-5)
    homography[2, 1] = generator.uniform(-7e-5, 7e-5)
    return homography

def warp_image(image, homography, generator):
    height, width = image.shape[:2]
    warped = cv2.warpPerspective(image, homography, (width, height),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    alpha = float(generator.uniform(0.82, 1.12))
    beta = float(generator.uniform(-12., 18.))
    return np.clip(warped.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

def project_points(points, homography):
    p = points.detach().cpu().numpy()
    p_h = np.concatenate([p, np.ones((len(p), 1), np.float32)], axis=1)
    projected = (homography @ p_h.T).T
    return torch.from_numpy((projected[:, :2] / projected[:, 2:3]).astype(np.float32))

def ground_truth_pairs(keypoints_1, keypoints_2, homography):
    if not len(keypoints_1) or not len(keypoints_2):
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    distances_12 = torch.cdist(project_points(keypoints_1, homography), keypoints_2)
    nearest_12 = distances_12.argmin(dim=1)
    errors = distances_12[torch.arange(len(keypoints_1)), nearest_12]
    distances_21 = torch.cdist(project_points(keypoints_2, np.linalg.inv(homography)), keypoints_1)
    nearest_21 = distances_21.argmin(dim=1)
    indices = torch.arange(len(keypoints_1))
    good = (nearest_21[nearest_12] == indices) & (errors < GEOMETRIC_THRESHOLD)
    return indices[good], nearest_12[good]

def build_pairs(source_images, warps_per_image, generator):
    pairs = []
    for image_index, image in enumerate(source_images):
        base = extract_features(image)
        for warp_index in range(warps_per_image):
            h = make_random_homography(*image.shape[:2], generator)
            warped_image = warp_image(image, h, generator)
            warped = extract_features(warped_image)
            i, j = ground_truth_pairs(base['keypoints'], warped['keypoints'], h)
            if len(i) > MAX_POSITIVES_PER_PAIR:
                keep = torch.randperm(len(i))[:MAX_POSITIVES_PER_PAIR]
                i, j = i[keep], j[keep]
            if len(i) < 32:
                print(f'Warning: image {image_index}, warp {warp_index}: {len(i)} positive pairs')
            pairs.append({'d1': base['descriptors'], 'd2': warped['descriptors'],
                          'k1': base['keypoints'], 'k2': warped['keypoints'],
                          'pos1': i, 'pos2': j, 'H': h, 'image1': image, 'image2': warped_image})
    return pairs

start = time.perf_counter()
calibration_pairs = build_pairs(calibration_images, CALIBRATION_WARPS_PER_IMAGE, rng)
validation_pairs = build_pairs(validation_images, VALIDATION_WARPS_PER_IMAGE, rng)
calibration = torch.cat([torch.cat([p['d1'], p['d2']], dim=0) for p in calibration_pairs], dim=0)
if len(calibration) > 50000:
    calibration = calibration[torch.randperm(len(calibration))[:50000]]
print('Feature extraction time (calibration and validation):', round(time.perf_counter() - start, 2), 's')
print('Calibration pairs:', len(calibration_pairs), 'Validation pairs:', len(validation_pairs),
      'Calibration descriptors:', len(calibration))

Feature extraction time (calibration and validation): 53.61 s
Calibration pairs: 108 Validation pairs: 18 Calibration descriptors: 50000


In [3]:
real_pairs = []
if RUN_HPatches:
    HPATCHES_URL = 'https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'
    HPATCHES_SEQUENCES_PER_TYPE = 6  # 6 viewpoint + 6 illumination scenes
    HPATCHES_TARGETS = (2, 3)       # two real photos per sequence
    hpatches_zip = data_dir / 'hpatches-sequences-release.zip'
    download_archive(HPATCHES_URL, hpatches_zip)

    def decode_hpatches_with_scale(raw_bytes):
        original = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if original is None:
            raise RuntimeError('Failed to decode HPatches PPM image')
        old_h, old_w = original.shape[:2]
        resized = decode_and_resize(raw_bytes)
        new_h, new_w = resized.shape[:2]
        scale = np.diag([new_w / old_w, new_h / old_h, 1.0]).astype(np.float64)
        return resized, scale

    real_pairs = []
    with zipfile.ZipFile(hpatches_zip) as archive:
        entries = set(archive.namelist())
        folders = sorted({name.rsplit('/', 1)[0] for name in entries if name.endswith('/1.ppm')})
        for prefix in ('v_', 'i_'):
            selected_folders = [folder for folder in folders if folder.split('/')[-1].startswith(prefix)]
            if len(selected_folders) < HPATCHES_SEQUENCES_PER_TYPE:
                raise RuntimeError(f'Expected more HPatches {prefix} folders; found {len(selected_folders)}')
            for folder in selected_folders[:HPATCHES_SEQUENCES_PER_TYPE]:
                image_1, scale_1 = decode_hpatches_with_scale(archive.read(folder + '/1.ppm'))
                features_1 = extract_features(image_1)
                for target_id in HPATCHES_TARGETS:
                    image_name = f'{folder}/{target_id}.ppm'
                    h_name = f'{folder}/H_1_{target_id}'
                    if image_name not in entries or h_name not in entries:
                        raise RuntimeError(f'Missing HPatches image or homography: {folder}, {target_id}')
                    image_2, scale_2 = decode_hpatches_with_scale(archive.read(image_name))
                    features_2 = extract_features(image_2)
                    original_h = np.loadtxt(io.BytesIO(archive.read(h_name))).reshape(3, 3)
                    resized_h = scale_2 @ original_h @ np.linalg.inv(scale_1)
                    real_pairs.append({
                        'name': f'{folder.split("/")[-1]}:1->{target_id}',
                        'group': prefix, 'd1': features_1['descriptors'],
                        'd2': features_2['descriptors'],
                        'k1': features_1['keypoints'], 'k2': features_2['keypoints'],
                        'H': resized_h, 'image1': image_1, 'image2': image_2,
                    })
    print('HPatches real-photo pairs:', len(real_pairs), '(viewpoint and illumination)')



Using persistent Drive cache: /content/drive/MyDrive/Colab Notebooks/CSE527_datasets/hpatches-sequences-release.zip
HPatches real-photo pairs: 24 (viewpoint and illumination)


In [4]:
import inspect, hashlib
OFFICIAL_MIN_COSSIM = -1  # match_xfeat default, NOT standalone match default 0.82
signature=inspect.signature(xfeat.match_xfeat)
if signature.parameters['min_cossim'].default != OFFICIAL_MIN_COSSIM:
    raise RuntimeError('Official match_xfeat default changed; review comparison settings.')
official_source=inspect.getsource(type(xfeat).match)
extract_source=inspect.getsource(type(xfeat).detectAndCompute)
hub_path=Path(inspect.getfile(type(xfeat))).resolve().parent.parent
sha_run=subprocess.run(['git','-C',str(hub_path),'rev-parse','HEAD'],capture_output=True,text=True)
xfeat_revision=sha_run.stdout.strip() if sha_run.returncode==0 else None
xfeat_provenance=dict(hub_path=str(hub_path),git_revision=xfeat_revision,
    match_sha256=hashlib.sha256(official_source.encode()).hexdigest(),
    extract_sha256=hashlib.sha256(extract_source.encode()).hexdigest(),
    match_signature=str(inspect.signature(xfeat.match)),match_xfeat_signature=str(signature))
print('Official XFeat provenance:',xfeat_provenance)
print('Official matcher source SHA256:',xfeat_provenance['match_sha256'])

@torch.inference_mode()
def official_match_cpu(p):
    # Direct call, no replacement of matmul, max, MNN or threshold logic.
    i,j=xfeat.match(p['d1'],p['d2'],min_cossim=OFFICIAL_MIN_COSSIM)
    return i.cpu().numpy(),j.cpu().numpy()



Official XFeat provenance: {'hub_path': '/root/.cache/torch/hub/verlab_accelerated_features_main', 'git_revision': None, 'match_sha256': 'e987220ff12fe3352c01ce2d411cdf70f2756f4039398eb2e98f62dfdcd0ad14', 'extract_sha256': '810bafd27792ba820c54ca66a11cd6eb243384641419436b39aeff5835ae9308', 'match_signature': '(feats1, feats2, min_cossim=0.82)', 'match_xfeat_signature': '(img1, img2, top_k=None, min_cossim=-1)'}
Official matcher source SHA256: e987220ff12fe3352c01ce2d411cdf70f2756f4039398eb2e98f62dfdcd0ad14


## 8bin·9bin 커널과 세 가지 비교 방식


In [5]:
import ctypes, tempfile, platform, json
CPU_THREADS = NUMBA_THREADS
ptr=ctypes.c_void_p;integer=ctypes.c_int
def adr(x):return x.ctypes.data_as(ptr)
AVX2 = True  # constructor checks CPU support before any SIMD call
cpp_source = '#include <cstdint>\n#include <vector>\n#include <algorithm>\n#include <immintrin.h>\n#include <omp.h>\nextern "C" int has_avx(){return __builtin_cpu_supports("avx512f") && __builtin_cpu_supports("avx512vpopcntdq");}\n__attribute__((target("popcnt"))) static inline int scalar(const uint64_t*a,const uint64_t*b){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[k]^b[k]);return d;}\n__attribute__((target("avx512f,avx512vpopcntdq"))) static int wide(const uint64_t*a,const uint64_t*b){\n __m512i x=_mm512_maskz_loadu_epi64(0x7f,a),y=_mm512_maskz_loadu_epi64(0x7f,b);\n return (int)_mm512_reduce_add_epi64(_mm512_popcnt_epi64(_mm512_xor_si512(x,y)));\n}\nextern "C" void encode_pack(const float*x,const float*e,int n,uint8_t*c,uint64_t*p){\n for(int i=0;i<n;i++){for(int k=0;k<7;k++)p[i*7+k]=0;\n for(int d=0;d<64;d++){int v=0;for(int k=0;k<7;k++)v+=x[i*64+d]>e[d*7+k];c[i*64+d]=v;\n for(int k=0;k<v;k++)p[i*7+k]|=uint64_t(1)<<d;}}\n}\nextern "C" void matrix(const uint64_t*a,const uint64_t*b,int n,int m,int threads,uint16_t*out){\n #pragma omp parallel for num_threads(threads) schedule(static)\n for(int i=0;i<n;i++)for(int j=0;j<m;j++)out[i*m+j]=scalar(a+i*7,b+j*7);\n}\nextern "C" void fused(const uint64_t*a,const uint64_t*b,int n,int m,int mode,int threads,int*ab,int*ba){\n if(n==0||m==0)return;\n const bool avx=mode==1&&has_avx();\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\n #pragma omp parallel num_threads(threads)\n {\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\n #pragma omp for schedule(static)\n for(int i=0;i<n;i++){int best=449,bj=m;\n for(int j=0;j<m;j++){int dist=0;\n if(mode==2){for(int k=0;k<7;k++){dist+=__builtin_popcountll(a[i*7+k]^b[j*7+k]);\n // Strict inequality preserves the smallest-index tie rule in both directions.\n if(dist>best&&dist>ds[j])break;}}\n else dist=avx?wide(a+i*7,b+j*7):scalar(a+i*7,b+j*7);\n if(dist<best){best=dist;bj=j;}\n if(dist<ds[j]||(dist==ds[j]&&i<ids[j])){ds[j]=dist;ids[j]=i;}\n }ab[i]=bj;}\n }\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){\n int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\n}\nextern "C" int has_avx2(){return __builtin_cpu_supports("avx2");}\nextern "C" void direct_scalar(const float*x,const float*et,int n,uint64_t*p){\n for(int i=0;i<n;i++)for(int k=0;k<7;k++){uint64_t w=0;for(int d=0;d<64;d++)w|=uint64_t(x[i*64+d]>et[k*64+d])<<d;p[i*7+k]=w;}\n}\nextern "C" __attribute__((target("avx2"))) void direct_avx(const float*x,const float*et,int n,uint64_t*p){\n for(int i=0;i<n;i++)for(int k=0;k<7;k++){uint64_t w=0;\n for(int d=0;d<64;d+=8){auto a=_mm256_loadu_ps(x+i*64+d),b=_mm256_loadu_ps(et+k*64+d);\n w|=uint64_t(uint32_t(_mm256_movemask_ps(_mm256_cmp_ps(a,b,_CMP_GT_OQ))))<<d;}p[i*7+k]=w;}\n}\n__attribute__((target("avx2"))) static inline __m256i counts4(__m256i v){\n const __m256i lut=_mm256_setr_epi8(0,1,1,2,1,2,2,3,1,2,2,3,2,3,3,4,0,1,1,2,1,2,2,3,1,2,2,3,2,3,3,4);\n auto mask=_mm256_set1_epi8(15);\n auto lo=_mm256_shuffle_epi8(lut,_mm256_and_si256(v,mask));\n auto hi=_mm256_shuffle_epi8(lut,_mm256_and_si256(_mm256_srli_epi16(v,4),mask));\n return _mm256_sad_epu8(_mm256_add_epi8(lo,hi),_mm256_setzero_si256());\n}\nextern "C" void transpose7(const uint64_t*p,int n,uint64_t*s){for(int k=0;k<7;k++)for(int i=0;i<n;i++)s[k*n+i]=p[i*7+k];}\nextern "C" __attribute__((target("avx2"))) void fused4(const uint64_t*a,const uint64_t*bs,int n,int m,int threads,int*ab,int*ba){\n if(n==0||m==0)return;\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\n #pragma omp parallel num_threads(threads)\n {\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\n #pragma omp for schedule(static)\n for(int i=0;i<n;i++){int best=449,bj=m;int j=0;\n for(;j+4<=m;j+=4){auto sum=_mm256_setzero_si256();\n for(int k=0;k<7;k++){auto av=_mm256_set1_epi64x(a[i*7+k]);auto bv=_mm256_loadu_si256((const __m256i*)(bs+k*m+j));sum=_mm256_add_epi64(sum,counts4(_mm256_xor_si256(av,bv)));}\n uint64_t dist[4];_mm256_storeu_si256((__m256i*)dist,sum);\n for(int l=0;l<4;l++){int d=dist[l],q=j+l;if(d<best){best=d;bj=q;}if(d<ds[q]||(d==ds[q]&&i<ids[q])){ds[q]=d;ids[q]=i;}}\n }\n for(;j<m;j++){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[i*7+k]^bs[k*m+j]);if(d<best){best=d;bj=j;}if(d<ds[j]||(d==ds[j]&&i<ids[j])){ds[j]=d;ids[j]=i;}}\n ab[i]=bj;}\n }\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\n}\n'
class ThermometerVariant:
 def __init__(self, bins):
  self.bins=bins;self.planes=bins-1;self.bytes_per_kp=self.planes*8
  self.name=f'{bins} bins / {self.planes} bits: method 1+2+3 SIMD4' if AVX2 else f'{bins} bins / {self.planes} bits: scalar fallback'
  self.edges=np.ascontiguousarray(torch.quantile(calibration,torch.arange(1,bins).float()/bins,dim=0).T.numpy(),dtype=np.float32)
  self.et=np.ascontiguousarray(self.edges.T)
  import re
  source=re.sub(r'\b7\b',str(self.planes),cpp_source).replace('449',str(64*self.planes+1))
  # Original AVX512 distance loads at most eight planes: use exact scalar here.
  # The benchmarked SIMD4 function is AVX2 and still processes ALL planes.
  begin=source.index(' __m512i x=');end=source.index('\n}',begin)
  source=source[:begin]+' return scalar(a,b);'+source[end:]
  folder=Path(tempfile.mkdtemp(prefix=f'thermometer_{bins}bin_'))
  (folder/'kernel.cpp').write_text(source)
  subprocess.run(['g++','-O3','-std=c++17','-shared','-fPIC','-fopenmp','-mpopcnt',str(folder/'kernel.cpp'),'-o',str(folder/'kernel.so')],check=True)
  self.lib=ctypes.CDLL(str(folder/'kernel.so'))
  self.lib.has_avx2.restype=integer
  if not self.lib.has_avx2():raise RuntimeError('This success method requires an AVX2 x86 CPU. Use a compatible CPU runtime.')
  for name in ('direct_scalar','direct_avx'):getattr(self.lib,name).argtypes=[ptr,ptr,integer,ptr]
  self.lib.transpose7.argtypes=[ptr,integer,ptr]
  self.lib.fused4.argtypes=[ptr,ptr,integer,integer,integer,ptr,ptr]
  self.lib.fused.argtypes=[ptr,ptr,integer,integer,integer,integer,ptr,ptr]
  self.lib.matrix.argtypes=[ptr,ptr,integer,integer,integer,ptr]
  for name in ('direct_scalar','direct_avx','transpose7','fused4','fused','matrix'):getattr(self.lib,name).restype=None
 def codes(self,x):return np.ascontiguousarray((x[:,:,None]>self.edges[None,:,:]).sum(axis=2),dtype=np.uint8)
 def encode(self,x,simd=True):
  x=np.ascontiguousarray(x,dtype=np.float32);out=np.empty((len(x),self.planes),np.uint64)
  fn=self.lib.direct_avx if simd and AVX2 else self.lib.direct_scalar
  fn(adr(x),adr(self.et),len(x),adr(out));return out
 def soa(self,p):
  out=np.empty((self.planes,len(p)),np.uint64);self.lib.transpose7(adr(p),len(p),adr(out));return out
 def match(self,a,bs):
  nn,mm=len(a),bs.shape[1]
  if not nn or not mm:return np.empty(0,np.int64),np.empty(0,np.int64)
  ab=np.empty(nn,np.int32);ba=np.empty(mm,np.int32)
  if AVX2:self.lib.fused4(adr(a),adr(bs),nn,mm,CPU_THREADS,adr(ab),adr(ba))
  else:
   b=np.ascontiguousarray(bs.T);self.lib.fused(adr(a),adr(b),nn,mm,0,CPU_THREADS,adr(ab),adr(ba))
  ii=np.arange(nn);keep=ba[ab]==ii;return ii[keep],ab[keep].astype(np.int64)
 def matrix(self,a,b):
  out=np.empty((len(a),len(b)),np.uint16);self.lib.matrix(adr(a),adr(b),len(a),len(b),CPU_THREADS,adr(out));return out
 def full(self,p):return self.match(self.encode(p['x']),self.soa(self.encode(p['y'])))


@njit(parallel=True, cache=False)
def pairwise_uint8_abs(code_1, code_2):
    out = np.empty((code_1.shape[0], code_2.shape[0]), np.uint16)
    for i in prange(code_1.shape[0]):
        for j in range(code_2.shape[0]):
            total = 0
            for k in range(code_1.shape[1]):
                a, b = int(code_1[i, k]), int(code_2[j, k])
                total += a - b if a >= b else b - a
            out[i, j] = total
    return out

def mutual_from_scores(scores, higher_is_better):
    if not scores.shape[0] or not scores.shape[1]:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
    if higher_is_better:
        nearest_12, nearest_21 = scores.argmax(axis=1), scores.argmax(axis=0)
    else:
        nearest_12, nearest_21 = scores.argmin(axis=1), scores.argmin(axis=0)
    indices = np.arange(scores.shape[0])
    keep = nearest_21[nearest_12] == indices
    return indices[keep], nearest_12[keep]


variants={bins:ThermometerVariant(bins) for bins in (8,9)}
for bins,variant in variants.items():
 # Diagnostic tests only; no other matcher is benchmarked.
 test_rng=np.random.default_rng(SEED)
 for nn,mm in ((0,3),(3,0),(1,1),(19,23),(9,3),(13,8),(800,800)):
  for tied in (False,True):
   x=test_rng.normal(size=(nn,64)).astype(np.float32);y=test_rng.normal(size=(mm,64)).astype(np.float32)
   if tied:x[:]=0;y[:]=0
   pa,pb=variant.encode(x),variant.encode(y)
   np.testing.assert_array_equal(pa,variant.encode(x,False))
   np.testing.assert_array_equal(pb,variant.encode(y,False))
   rd=pairwise_uint8_abs(variant.codes(x),variant.codes(y))
   np.testing.assert_array_equal(rd,variant.matrix(pa,pb))
   for u,v in zip(mutual_from_scores(rd,False),variant.match(pa,variant.soa(pb))):np.testing.assert_array_equal(u,v)
 for x in (variant.et,np.nextafter(variant.et,-np.inf),np.nextafter(variant.et,np.inf)):
  packed=variant.encode(x);np.testing.assert_array_equal(packed,variant.encode(x,False))
  ref=np.zeros((len(x),variant.planes),np.uint64);codes=variant.codes(x)
  for k in range(variant.planes):
   for d in range(64):ref[:,k]|=(codes[:,d]>k).astype(np.uint64)<<np.uint64(d)
  np.testing.assert_array_equal(packed,ref)
 print('PASS: ABS distance, MNN, direct encoding, threshold equality, ties, empty inputs, candidate tails.')




from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import sklearn
from threadpoolctl import threadpool_limits, threadpool_info
mh_thread_limit=threadpool_limits(limits=CPU_THREADS)
MH_SOURCE_URL='https://cs.nju.edu.cn/lwj/paper/SIGIR12_MH.pdf'
mh_train=np.ascontiguousarray(calibration.numpy(),dtype=np.float32)
if len(mh_train)<64 or not np.isfinite(mh_train).all():
 raise ValueError('PCA-MQ requires at least 64 finite calibration descriptors.')
start=time.perf_counter()
mh_pca=PCA(n_components=64,whiten=False,svd_solver='full').fit(mh_train)
mh_mean=np.ascontiguousarray(mh_pca.mean_,dtype=np.float32)
mh_components=np.ascontiguousarray(mh_pca.components_,dtype=np.float32)
mh_projected=np.ascontiguousarray((mh_train-mh_mean)@mh_components.T)
mh_pca_fit_s=time.perf_counter()-start

def fit_mh_edges(train,bins,label):
 start=time.perf_counter();centers=np.empty((64,bins),np.float32)
 for d in range(64):
  if len(np.unique(train[:,d]))<bins:raise ValueError(f'{label}: too few distinct values in dimension {d}')
  km=KMeans(n_clusters=bins,init='k-means++',n_init=10,max_iter=300,tol=1e-4,
            random_state=SEED+d,algorithm='lloyd').fit(train[:,d:d+1])
  centers[d]=np.sort(km.cluster_centers_[:,0])
  if np.any(np.diff(centers[d])<=0):raise ValueError(f'{label}: duplicate centers in dimension {d}')
 edges=np.ascontiguousarray((centers[:,:-1]+centers[:,1:])*0.5)
 return edges,centers,time.perf_counter()-start

mh_parameters={}
for b in (8,9):
 pe,pc,pt=fit_mh_edges(mh_projected,b,f'PCA-MQ {b}bin')
 ie,ic,it=fit_mh_edges(mh_train,b,f'identity k-means {b}bin')
 mh_parameters[b]=dict(pca_edges=pe,pca_centers=pc,pca_kmeans_fit_s=pt,
   identity_edges=ie,identity_centers=ic,identity_kmeans_fit_s=it)
 print('Calibration only:',b,'bins; PCA k-means:',round(pt,3),'s; no-projection k-means:',round(it,3),'s')
print('PCA fit seconds:',mh_pca_fit_s)
mh_cpp_source='\n#include <cstdint>\n#include <vector>\n#include <immintrin.h>\n#include <omp.h>\nextern "C" void packed_encode(const float*x,const float*et,int n,int planes,int bits,uint8_t*out){\n const int bytes=bits*8;\n for(int i=0;i<n;i++)for(int g=0;g<8;g++){\n  auto a=_mm256_loadu_ps(x+i*64+g*8);auto bins=_mm256_setzero_si256();\n  for(int k=0;k<planes;k++){\n   auto e=_mm256_loadu_ps(et+k*64+g*8);\n   bins=_mm256_sub_epi32(bins,_mm256_castps_si256(_mm256_cmp_ps(a,e,_CMP_GT_OQ)));\n  }\n  uint32_t v[8];_mm256_storeu_si256((__m256i*)v,bins);\n  uint32_t word=0;for(int k=0;k<8;k++)word|=v[k]<<(k*bits);\n  uint8_t*p=out+i*bytes+g*bits;for(int b=0;b<bits;b++)p[b]=(word>>(b*8))&255;\n }\n}\nstatic void decode(const uint8_t*p,int n,int bits,uint8_t*c){\n const int bytes=bits*8,mask=(1<<bits)-1;\n for(int i=0;i<n;i++)for(int g=0;g<8;g++){\n  uint32_t word=0;for(int b=0;b<bits;b++)word|=uint32_t(p[i*bytes+g*bits+b])<<(b*8);\n  for(int k=0;k<8;k++)c[i*64+g*8+k]=(word>>(k*bits))&mask;\n }\n}\nstatic int distance(const uint8_t*a,const uint8_t*b){\n auto sum=_mm256_setzero_si256();\n for(int k=0;k<64;k+=32){\n  auto x=_mm256_loadu_si256((const __m256i*)(a+k)),y=_mm256_loadu_si256((const __m256i*)(b+k));\n  sum=_mm256_add_epi64(sum,_mm256_sad_epu8(x,y));\n }\n uint64_t v[4];_mm256_storeu_si256((__m256i*)v,sum);\n return int(v[0]+v[1]+v[2]+v[3]);\n}\nextern "C" void packed_matrix(const uint8_t*a,const uint8_t*b,int n,int m,int bits,int threads,uint16_t*out){\n std::vector<uint8_t> aa(n*64),bb(m*64);decode(a,n,bits,aa.data());decode(b,m,bits,bb.data());\n #pragma omp parallel for num_threads(threads) schedule(static)\n for(int i=0;i<n;i++)for(int j=0;j<m;j++)out[i*m+j]=distance(aa.data()+i*64,bb.data()+j*64);\n}\nextern "C" void packed_mnn(const uint8_t*a,const uint8_t*b,int n,int m,int bits,int threads,int*ab,int*ba){\n if(!n||!m)return;\n std::vector<uint8_t> aa(n*64),bb(m*64);decode(a,n,bits,aa.data());decode(b,m,bits,bb.data());\n std::vector<int> cd(threads*m,513),ci(threads*m,n);\n #pragma omp parallel num_threads(threads)\n {\n  int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\n  #pragma omp for schedule(static)\n  for(int i=0;i<n;i++){\n   int best=513,bj=m;\n   for(int j=0;j<m;j++){\n    int d=distance(aa.data()+i*64,bb.data()+j*64);\n    if(d<best){best=d;bj=j;}\n    if(d<ds[j]||(d==ds[j]&&i<ids[j])){ds[j]=d;ids[j]=i;}\n   }ab[i]=bj;\n  }\n }\n for(int j=0;j<m;j++){\n  int best=513,bi=n;\n  for(int t=0;t<threads;t++){int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}\n  ba[j]=bi;\n }\n}\n'
mh_folder=Path(tempfile.mkdtemp(prefix='mh_8_9_'))
(mh_folder/'kernel.cpp').write_text(mh_cpp_source)
subprocess.run(['g++','-O3','-std=c++17','-shared','-fPIC','-fopenmp','-mavx2',
 str(mh_folder/'kernel.cpp'),'-o',str(mh_folder/'kernel.so')],check=True)
mh_lib=ctypes.CDLL(str(mh_folder/'kernel.so'))
mh_lib.packed_encode.argtypes=[ptr,ptr,integer,integer,integer,ptr]
mh_lib.packed_matrix.argtypes=[ptr,ptr,integer,integer,integer,integer,ptr]
mh_lib.packed_mnn.argtypes=[ptr,ptr,integer,integer,integer,integer,ptr,ptr]
for fn in ('packed_encode','packed_matrix','packed_mnn'):getattr(mh_lib,fn).restype=None

class ManhattanVariant:
 def __init__(self,name,bins,edges,projection,encoding,boundary_method,role):
  self.name=name;self.bins=bins;self.edges=edges;self.projection=projection;self.encoding=encoding
  self.boundary_method=boundary_method;self.role=role;self.index_bits=(bins-1).bit_length()
  self.planes=bins-1 if encoding=='thermometer' else None
  self.bytes_per_kp=(self.planes if self.planes else self.index_bits)*8
  self.backend=variants[bins];self.et=np.ascontiguousarray(edges.T)
 def project(self,x):
  x=np.ascontiguousarray(x,dtype=np.float32)
  return np.ascontiguousarray((x-mh_mean)@mh_components.T) if self.projection=='PCA64' else x
 def codes(self,x):
  z=self.project(x)
  return np.ascontiguousarray((z[:,:,None]>self.edges[None,:,:]).sum(2),dtype=np.uint8)
 @staticmethod
 def pack_codes(c,bits):
  out=np.empty((len(c),bits*8),np.uint8)
  for g in range(8):
   w=np.zeros(len(c),np.uint32)
   for k in range(8):w|=c[:,g*8+k].astype(np.uint32)<<np.uint32(k*bits)
   for b in range(bits):out[:,g*bits+b]=((w>>np.uint32(b*8))&255).astype(np.uint8)
  return out
 @staticmethod
 def unpack_codes(p,bits):
  out=np.empty((len(p),64),np.uint8)
  for g in range(8):
   w=np.zeros(len(p),np.uint32)
   for b in range(bits):w|=p[:,g*bits+b].astype(np.uint32)<<np.uint32(b*8)
   for k in range(8):out[:,g*8+k]=((w>>np.uint32(k*bits))&((1<<bits)-1)).astype(np.uint8)
  return out
 def encode(self,x):
  z=self.project(x)
  if self.planes:
   out=np.empty((len(z),self.planes),np.uint64)
   self.backend.lib.direct_avx(adr(z),adr(self.et),len(z),adr(out));return out
  out=np.empty((len(z),self.bytes_per_kp),np.uint8)
  mh_lib.packed_encode(adr(z),adr(self.et),len(z),self.bins-1,self.index_bits,adr(out));return out
 def soa(self,p):return self.backend.soa(p) if self.planes else p
 def match(self,a,b):
  if self.planes:return self.backend.match(a,b)
  n,m=len(a),len(b)
  if not n or not m:return np.empty(0,np.int64),np.empty(0,np.int64)
  ab=np.empty(n,np.int32);ba=np.empty(m,np.int32)
  mh_lib.packed_mnn(adr(a),adr(b),n,m,self.index_bits,CPU_THREADS,adr(ab),adr(ba))
  i=np.arange(n);keep=ba[ab]==i;return i[keep],ab[keep].astype(np.int64)
 def matrix(self,a,b):
  if self.planes:return self.backend.matrix(a,b)
  out=np.empty((len(a),len(b)),np.uint16)
  mh_lib.packed_matrix(adr(a),adr(b),len(a),len(b),self.index_bits,CPU_THREADS,adr(out));return out
 def full(self,p):return self.match(self.encode(p['x']),self.soa(self.encode(p['y'])))

mh_variants={};quantile_nbc_names={};mh_names={};identity_names={}
for b in (8,9):
 bits=(b-1).bit_length();pars=mh_parameters[b]
 identity=f'1: {b}bin no projection kmeans thermometer SIMD4'
 mh=f'2: {b}bin MH PCA64 kmeans NBC{bits} ABS'+(' (9-bin extension)' if b==9 else ' (paper PCA-MQ)')
 quantile=f'3: {b}bin no projection quantile NBC{bits} ABS'
 identity_names[b]=identity;mh_names[b]=mh;quantile_nbc_names[b]=quantile
 mh_variants[identity]=ManhattanVariant(identity,b,pars['identity_edges'],'none','thermometer','k-means','1')
 mh_variants[mh]=ManhattanVariant(mh,b,pars['pca_edges'],'PCA64',f'NBC{bits}','k-means','2')
 mh_variants[quantile]=ManhattanVariant(quantile,b,variants[b].edges,'none',f'NBC{bits}','quantile','3')

check_rng=np.random.default_rng(SEED+321)
for name,v in mh_variants.items():
 for n,m in ((0,3),(3,0),(1,1),(19,23),(9,3),(800,800)):
  for tied in (False,True):
   x=check_rng.normal(size=(n,64)).astype(np.float32);y=check_rng.normal(size=(m,64)).astype(np.float32)
   if tied:x[:]=0;y[:]=0
   a,b=v.encode(x),v.encode(y);ca,cb=v.codes(x),v.codes(y)
   if not v.planes:
    np.testing.assert_array_equal(v.unpack_codes(a,v.index_bits),ca)
    np.testing.assert_array_equal(v.unpack_codes(b,v.index_bits),cb)
   ref=pairwise_uint8_abs(ca,cb)
   np.testing.assert_array_equal(v.matrix(a,b),ref)
   for actual,expected in zip(v.match(a,v.soa(b)),mutual_from_scores(ref,False)):
    np.testing.assert_array_equal(actual,expected)
 for z in (v.edges.T,np.nextafter(v.edges.T,-np.inf),np.nextafter(v.edges.T,np.inf)):
  z=np.ascontiguousarray(z)
  c=np.stack([np.searchsorted(v.edges[d],z[:,d],side='left') for d in range(64)],axis=1)
  direct=np.empty((len(z),v.index_bits*8),np.uint8)
  mh_lib.packed_encode(adr(z),adr(v.et),len(z),v.bins-1,v.index_bits,adr(direct))
  np.testing.assert_array_equal(v.unpack_codes(direct,v.index_bits),c)
 print('PASS exact ABS, MNN, ties, empty inputs, byte packing, threshold equality:',name)
for b in (8,9):
 v=mh_variants[mh_names[b]]
 c=np.repeat(np.arange(b,dtype=np.uint8)[:,None],64,axis=1)
 p=v.pack_codes(c,v.index_bits)
 np.testing.assert_array_equal(v.matrix(p,p),64*np.abs(np.arange(b)[:,None]-np.arange(b)[None,:]))
 pars=mh_parameters[b]
 for prefix in ('pca','identity'):
  np.testing.assert_array_equal(pars[prefix+'_edges'],(pars[prefix+'_centers'][:,:-1]+pars[prefix+'_centers'][:,1:])*0.5)
 # Same quantile edges, same ordinal distance, two different representations.
 x=check_rng.normal(size=(71,64)).astype(np.float32);y=check_rng.normal(size=(83,64)).astype(np.float32)
 q=mh_variants[quantile_nbc_names[b]];t=variants[b]
 np.testing.assert_array_equal(q.matrix(q.encode(x),q.encode(y)),t.matrix(t.encode(x),t.encode(y)))
 for actual,expected in zip(q.full(dict(x=x,y=y)),t.full(dict(x=x,y=y))):np.testing.assert_array_equal(actual,expected)
np.testing.assert_allclose(mh_projected,mh_pca.transform(mh_train),rtol=1e-4,atol=2e-6)
mh_provenance=dict(source=MH_SOURCE_URL,paper_variant='PCA-MQ for 8; generalized 9-bin MQ for 9',
 d=64,bins=[8,9],index_bits={'8':3,'9':4},hash_bits={'8':192,'9':256},
 pca=dict(center=True,whiten=False,solver='full',fit_seconds=mh_pca_fit_s),
 kmeans=dict(init='k-means++',n_init=10,max_iter=300,tol=1e-4,algorithm='lloyd',seed_rule='SEED + dimension',
   fit_seconds={str(b):{k:v for k,v in pars.items() if k.endswith('_s')} for b,pars in mh_parameters.items()}),
 calibration_descriptor_count=len(mh_train),calibration_sha256=hashlib.sha256(mh_train.tobytes()).hexdigest(),
 boundary_tie='smaller ordinal bin; > comparison',post_projection_renormalization=False,
 packed_format='8 ordinal indices per 3 or 4 bytes; little-endian NBC',
 matching='exact index L1, fused bidirectional MNN, smallest-index tie',
 implementation='C++ AVX2 direct NBC encode and uint8 SAD; unpack inside every match call; not paper source code',
 nbc_unpacked_scratch_bytes_per_kp=64,sklearn_version=sklearn.__version__,threadpools=threadpool_info())

OFFICIAL='official XFeat cosine MNN (-1)'
METHODS=[OFFICIAL]+[f'{b}bin / {b-1}bit AVX2 SIMD4' for b in variants]
BY_NAME={f'{b}bin / {b-1}bit AVX2 SIMD4':v for b,v in variants.items()}
BY_NAME.update(mh_variants)
METHODS+=list(mh_variants)
assert REPEATS % len(METHODS)==PIPELINE_REPEATS % len(METHODS)==0
print('Methods:',METHODS)
@torch.inference_mode()
def official_pipeline(t1,t2):return xfeat.match_xfeat(t1,t2,top_k=TOP_K,min_cossim=OFFICIAL_MIN_COSSIM)
@torch.inference_mode()
def compressed_pipeline(t1,t2,v):
 d1=xfeat.detectAndCompute(t1,top_k=TOP_K)[0];d2=xfeat.detectAndCompute(t2,top_k=TOP_K)[0]
 i,j=v.full(dict(x=d1['descriptors'].numpy(),y=d2['descriptors'].numpy()))
 return d1['keypoints'][i].numpy(),d2['keypoints'][j].numpy()
@torch.inference_mode()
def extraction_only(t1,t2):return xfeat.detectAndCompute(t1,top_k=TOP_K)[0],xfeat.detectAndCompute(t2,top_k=TOP_K)[0]
def prepare(p):
 x=np.ascontiguousarray(p['d1'].numpy(),dtype=np.float32);y=np.ascontiguousarray(p['d2'].numpy(),dtype=np.float32)
 assert np.isfinite(x).all() and np.isfinite(y).all()
 cached={name:(v.encode(x),v.soa(v.encode(y))) for name,v in BY_NAME.items()}
 return dict(p,x=x,y=y,cached=cached)
def cached_match(p,name):
 if name==OFFICIAL:return official_match_cpu(p)
 v=BY_NAME[name];return v.match(*p['cached'][name])
def total_match(p,name):
 if name==OFFICIAL:return official_match_cpu(p)
 return BY_NAME[name].full(p)


PASS: ABS distance, MNN, direct encoding, threshold equality, ties, empty inputs, candidate tails.
PASS: ABS distance, MNN, direct encoding, threshold equality, ties, empty inputs, candidate tails.
Calibration only: 8 bins; PCA k-means: 46.425 s; no-projection k-means: 49.807 s
Calibration only: 9 bins; PCA k-means: 49.287 s; no-projection k-means: 51.268 s
PCA fit seconds: 0.21941635600001064
PASS exact ABS, MNN, ties, empty inputs, byte packing, threshold equality: 1: 8bin no projection kmeans thermometer SIMD4
PASS exact ABS, MNN, ties, empty inputs, byte packing, threshold equality: 2: 8bin MH PCA64 kmeans NBC3 ABS (paper PCA-MQ)
PASS exact ABS, MNN, ties, empty inputs, byte packing, threshold equality: 3: 8bin no projection quantile NBC3 ABS
PASS exact ABS, MNN, ties, empty inputs, byte packing, threshold equality: 1: 9bin no projection kmeans thermometer SIMD4
PASS exact ABS, MNN, ties, empty inputs, byte packing, threshold equality: 2: 9bin MH PCA64 kmeans NBC4 ABS (9-bin extens

## 동일 디스크립터 정확도와 분위수 NBC/thermometer 동등성 검증


In [6]:
# Shared extracted descriptors: no per-method extraction in the matching benchmark.
datasets={'synthetic':[prepare(p) for p in validation_pairs]}
if real_pairs:datasets['HPatches']=[prepare(p) for p in real_pairs]
accuracy_results=[];pair_accuracy=[];pair_manifest=[]
for label,pairs in datasets.items():
 totals={name:dict(mutual=0,correct=0,errors=[]) for name in METHODS}
 for idx,p in enumerate(pairs):
  pair_manifest.append(dict(dataset=label,pair=idx,name=p.get('name',f'synthetic_{idx}'),
   n=len(p['d1']),m=len(p['d2']),homography=np.asarray(p['H']).tolist(),
   descriptor_sha256=hashlib.sha256(p['x'].tobytes()+p['y'].tobytes()).hexdigest()))
  for name in METHODS:
   i,j=cached_match(p,name)
   if name!=OFFICIAL:
    v=BY_NAME[name];rd=pairwise_uint8_abs(v.codes(p['x']),v.codes(p['y']))
    np.testing.assert_array_equal(rd,v.matrix(p['cached'][name][0],v.encode(p['y'])))
    for u,w in zip(mutual_from_scores(rd,False),(i,j)):np.testing.assert_array_equal(u,w)
   err=np.linalg.norm(project_points(p['k1'][i],p['H']).numpy()-p['k2'][j].numpy(),axis=1)
   good=int((err<GEOMETRIC_THRESHOLD).sum())
   t=totals[name];t['mutual']+=len(i);t['correct']+=good;t['errors'].extend(err.tolist())
   pair_accuracy.append(dict(dataset=label,pair=idx,method=name,mutual=len(i),correct=good,
    precision_pct=100*good/len(i) if len(i) else None,median_px=float(np.median(err)) if len(err) else None))
  for b in (8,9):
   q=mh_variants[quantile_nbc_names[b]];t=variants[b]
   np.testing.assert_array_equal(q.matrix(q.encode(p['x']),q.encode(p['y'])),t.matrix(t.encode(p['x']),t.encode(p['y'])))
   for a,z in zip(cached_match(p,quantile_nbc_names[b]),cached_match(p,f'{b}bin / {b-1}bit AVX2 SIMD4')):np.testing.assert_array_equal(a,z)
  print('Accuracy and quantile representation equivalence verified:',label,idx+1,'/',len(pairs))
 for name,t in totals.items():
  v=BY_NAME.get(name);errors=t.pop('errors')
  accuracy_results.append(dict(dataset=label,method=name,pairs=len(pairs),**t,
   precision_pct=100*t['correct']/t['mutual'] if t['mutual'] else None,
   median_px=float(np.median(errors)) if errors else None,descriptor_bytes=v.bytes_per_kp if v else 256,
   bins=v.bins if v else None,thermometer_bits=v.planes if v else None,exact_abs_verified=True if v else None,
   projection=getattr(v,'projection','none') if v else 'none',
   boundary_method=getattr(v,'boundary_method','quantile') if v else None,
   encoding=getattr(v,'encoding','thermometer') if v else 'float32',
   hash_bits=v.bytes_per_kp*8 if v else 2048,
   role=getattr(v,'role','baseline') if v else 'official',
   index_bits=v.index_bits if isinstance(v,ManhattanVariant) else None,
   paper_exact_bin_count=v.bins==8 if isinstance(v,ManhattanVariant) and v.role=='2' else None,
   unpack_scratch_bytes_per_kp=64 if isinstance(v,ManhattanVariant) and v.encoding.startswith('NBC') else 0))
display(pd.DataFrame(accuracy_results))


Accuracy and quantile representation equivalence verified: synthetic 1 / 18
Accuracy and quantile representation equivalence verified: synthetic 2 / 18
Accuracy and quantile representation equivalence verified: synthetic 3 / 18
Accuracy and quantile representation equivalence verified: synthetic 4 / 18
Accuracy and quantile representation equivalence verified: synthetic 5 / 18
Accuracy and quantile representation equivalence verified: synthetic 6 / 18
Accuracy and quantile representation equivalence verified: synthetic 7 / 18
Accuracy and quantile representation equivalence verified: synthetic 8 / 18
Accuracy and quantile representation equivalence verified: synthetic 9 / 18
Accuracy and quantile representation equivalence verified: synthetic 10 / 18
Accuracy and quantile representation equivalence verified: synthetic 11 / 18
Accuracy and quantile representation equivalence verified: synthetic 12 / 18
Accuracy and quantile representation equivalence verified: synthetic 13 / 18
Accuracy

,dataset,method,pairs,mutual,correct,precision_pct,median_px,descriptor_bytes,bins,thermometer_bits,exact_abs_verified,projection,boundary_method,encoding,hash_bits,role,index_bits,paper_exact_bin_count,unpack_scratch_bytes_per_kp
0,synthetic,official XFeat cosine MNN (-1),18,8200,5611,68.426829,1.366410,256,NaN,NaN,None,none,None,float32,2048,official,NaN,None,0
1,synthetic,8bin / 7bit AVX2 SIMD4,18,8098,5433,67.090640,1.415396,56,8.0,7.0,True,none,quantile,thermometer,448,baseline,NaN,None,0
2,synthetic,9bin / 8bit AVX2 SIMD4,18,8098,5482,67.695727,1.395179,64,9.0,8.0,True,none,quantile,thermometer,512,baseline,NaN,None,0
3,synthetic,1: 8bin no projection kmeans thermometer SIMD4,18,8035,5394,67.131301,1.405773,56,8.0,7.0,True,none,k-means,thermometer,448,1,3.0,None,0
4,synthetic,2: 8bin MH PCA64 kmeans NBC3 ABS (paper PCA-MQ),18,7979,5260,65.923048,1.463955,24,8.0,NaN,True,PCA64,k-means,NBC3,192,2,3.0,True,64
5,synthetic,3: 8bin no projection quantile NBC3 ABS,18,8098,5433,67.090640,1.415396,24,8.0,NaN,True,none,quantile,NBC3,192,3,3.0,None,64
6,synthetic,1: 9bin no projection kmeans thermometer SIMD4,18,8097,5412,66.839570,1.425821,64,9.0,8.0,True,none,k-means,thermometer,512,1,4.0,None,0
7,synthetic,2: 9bin MH PCA64 kmeans NBC4 ABS (9-bin extens...,18,8031,5301,66.006724,1.439910,32,9.0,NaN,True,PCA64,k-means,NBC4,256,2,4.0,False,64
8,synthetic,3: 9bin no projection quantile NBC4 ABS,18,8098,5482,67.695727,1.395179,32,9.0,NaN,True,none,quantile,NBC4,256,3,4.0,None,64
9,HPatches,official XFeat cosine MNN (-1),24,10607,7138,67.295182,1.176445,256,NaN,NaN,None,none,None,float32,2048,official,NaN,None,0


## 순서를 균등하게 섞어 번갈아 시간 측정

In [7]:
# Rotate randomized orders in blocks: each method occupies each position once/block.
# Timing RNG is independent of calibration and image RNGs.
raw_timing=[]
def balanced_orders(names,repeats,generator):
 assert repeats>0 and repeats%len(names)==0
 orders=[]
 for _ in range(repeats//len(names)):
  perm=list(generator.permutation(names));rotations=list(generator.permutation(len(names)))
  orders.extend([perm[k:]+perm[:k] for k in rotations])
 return orders

def interleaved_bench(funcs,label,idx,scope,repeats,seed):
 names=list(funcs);g=np.random.default_rng(seed)
 for _ in range(WARMUP):
  for name in g.permutation(names):funcs[name]()
 orders=balanced_orders(names,repeats,g)
 samples={name:[] for name in names}
 for round_id,order in enumerate(orders):
  for position,name in enumerate(order):
   begin=time.perf_counter_ns();result=funcs[name]();elapsed=(time.perf_counter_ns()-begin)/1e6
   samples[name].append(elapsed)
   raw_timing.append(dict(dataset=label,pair=int(idx),scope=scope,round=round_id,
    position=position,method=name,ms=elapsed))
   del result  # output disposal is outside the next timed call
 return samples

def stats(samples):
 return dict(median_ms=float(np.median(samples)),p10_ms=float(np.percentile(samples,10)),
  p90_ms=float(np.percentile(samples,90)))

timing_results=[];pipeline_results=[]
for label_id,(label,pairs) in enumerate(datasets.items()):
 indices=np.unique(np.linspace(0,len(pairs)-1,min(TIMING_PAIRS_PER_DATASET,len(pairs)),dtype=int))
 for idx in indices:
  p=pairs[idx];seed=TIMING_SEED+label_id*10000+int(idx)*100
  scopes=['total','cached']
  np.random.default_rng(seed).shuffle(scopes)
  measured={}
  for scope in scopes:
   fn=total_match if scope=='total' else cached_match
   funcs={name:(lambda name=name,fn=fn:fn(p,name)) for name in METHODS}
   measured[scope]=interleaved_bench(funcs,label,idx,scope,REPEATS,seed+(1 if scope=='total' else 2))
  official_total=float(np.median(measured['total'][OFFICIAL]))
  for name in METHODS:
   total=stats(measured['total'][name]);cached=stats(measured['cached'][name])
   # Paired ratios per round retain common runtime drift; summary also records pair-median ratios.
   ratios=np.asarray(measured['total'][OFFICIAL])/np.asarray(measured['total'][name])
   timing_results.append(dict(dataset=label,pair=int(idx),method=name,n=len(p['d1']),m=len(p['d2']),
    cached_ms=cached['median_ms'],total_ms=total['median_ms'],p10_ms=total['p10_ms'],p90_ms=total['p90_ms'],
    paired_speedup=official_total/total['median_ms'],round_speedup_median=float(np.median(ratios)),
    round_speedup_p10=float(np.percentile(ratios,10)),round_speedup_p90=float(np.percentile(ratios,90))))
  print('Matching timed:',label,int(idx),{name:round(np.median(measured['total'][name]),3) for name in METHODS})
  # Full pipeline is secondary: every method calls the same official extractor twice.
  t1=image_to_tensor(p['image1']);t2=image_to_tensor(p['image2'])
  d1,d2=extraction_only(t1,t2)
  funcs={OFFICIAL:lambda:official_pipeline(t1,t2)}
  for name,v in BY_NAME.items():funcs[name]=lambda v=v:compressed_pipeline(t1,t2,v)
  for name,fn in funcs.items():
   if name==OFFICIAL:i,j=xfeat.match(d1['descriptors'],d2['descriptors'],min_cossim=OFFICIAL_MIN_COSSIM)
   else:i,j=BY_NAME[name].full(dict(x=d1['descriptors'].numpy(),y=d2['descriptors'].numpy()))
   coords=fn()
   np.testing.assert_array_equal(coords[0],d1['keypoints'][i].numpy())
   np.testing.assert_array_equal(coords[1],d2['keypoints'][j].numpy())
  ps=interleaved_bench(funcs,label,idx,'pipeline',PIPELINE_REPEATS,seed+3)
  official_ms=float(np.median(ps[OFFICIAL]))
  for name in METHODS:
   t=stats(ps[name]);pipeline_results.append(dict(dataset=label,pair=int(idx),method=name,
    pipeline_ms=t['median_ms'],p10_ms=t['p10_ms'],p90_ms=t['p90_ms'],paired_speedup=official_ms/t['median_ms']))
  print('Full pipeline timed:',label,int(idx))
frame=pd.DataFrame(timing_results)
speed_summary=frame.groupby(['dataset','method'],sort=False)[['cached_ms','total_ms','p10_ms','p90_ms','paired_speedup','round_speedup_median']].median().reset_index()
pipeline_summary=pd.DataFrame(pipeline_results).groupby(['dataset','method'],sort=False)[['pipeline_ms','p10_ms','p90_ms','paired_speedup']].median().reset_index()
comparison=pd.DataFrame(accuracy_results).merge(speed_summary,on=['dataset','method']).merge(
 pipeline_summary[['dataset','method','pipeline_ms','paired_speedup']].rename(columns={'paired_speedup':'pipeline_speedup'}),on=['dataset','method'])
for label in comparison.dataset.unique():
 mask=comparison.dataset==label
 base=comparison.loc[mask&(comparison.method==OFFICIAL)].iloc[0]
 comparison.loc[mask,'precision_gap_pp']=comparison.loc[mask,'precision_pct']-base.precision_pct
 comparison.loc[mask,'descriptor_reduction_pct']=100*(1-comparison.loc[mask,'descriptor_bytes']/256)
print('PRIMARY: same descriptors, projection+encoding+packing/layout+MNN. Times: median of per-pair medians.')
print('paired_speedup: median of official/method per-pair median ratios; >1 means faster.')
display(comparison[['dataset','method','precision_pct','precision_gap_pp','correct','mutual','descriptor_bytes',
 'projection','boundary_method','encoding','total_ms','paired_speedup','p10_ms','p90_ms','pipeline_ms','pipeline_speedup']].round(4))
print('SECONDARY: extraction-inclusive pipeline, directly measured with interleaved method order.')
display(pipeline_summary.round(4))
print('cached_ms is measured separately and may exceed total_ms under runtime noise; inspect raw samples.')


Matching timed: synthetic 0 {'official XFeat cosine MNN (-1)': np.float64(17.675), '8bin / 7bit AVX2 SIMD4': np.float64(9.399), '9bin / 8bit AVX2 SIMD4': np.float64(8.05), '1: 8bin no projection kmeans thermometer SIMD4': np.float64(8.013), '2: 8bin MH PCA64 kmeans NBC3 ABS (paper PCA-MQ)': np.float64(12.314), '3: 8bin no projection quantile NBC3 ABS': np.float64(9.993), '1: 9bin no projection kmeans thermometer SIMD4': np.float64(10.114), '2: 9bin MH PCA64 kmeans NBC4 ABS (9-bin extension)': np.float64(14.868), '3: 9bin no projection quantile NBC4 ABS': np.float64(8.648)}
Full pipeline timed: synthetic 0
Matching timed: synthetic 4 {'official XFeat cosine MNN (-1)': np.float64(20.835), '8bin / 7bit AVX2 SIMD4': np.float64(11.194), '9bin / 8bit AVX2 SIMD4': np.float64(11.977), '1: 8bin no projection kmeans thermometer SIMD4': np.float64(9.991), '2: 8bin MH PCA64 kmeans NBC3 ABS (paper PCA-MQ)': np.float64(24.984), '3: 8bin no projection quantile NBC3 ABS': np.float64(8.972), '1: 9bin n

,dataset,method,precision_pct,precision_gap_pp,correct,mutual,descriptor_bytes,projection,boundary_method,encoding,total_ms,paired_speedup,p10_ms,p90_ms,pipeline_ms,pipeline_speedup
0,synthetic,official XFeat cosine MNN (-1),68.4268,0.0000,5611,8200,256,none,None,float32,10.2546,1.0000,8.0259,16.7764,207.3677,1.0000
1,synthetic,8bin / 7bit AVX2 SIMD4,67.0906,-1.3362,5433,8098,56,none,quantile,thermometer,4.7736,2.0160,3.7523,8.0242,234.1249,0.9909
2,synthetic,9bin / 8bit AVX2 SIMD4,67.6957,-0.7311,5482,8098,64,none,quantile,thermometer,4.5176,1.9476,4.0280,10.7891,241.5989,0.8774
3,synthetic,1: 8bin no projection kmeans thermometer SIMD4,67.1313,-1.2955,5394,8035,56,none,k-means,thermometer,6.3831,2.0794,3.7893,7.9783,207.2498,1.0006
4,synthetic,2: 8bin MH PCA64 kmeans NBC3 ABS (paper PCA-MQ),65.9230,-2.5038,5260,7979,24,PCA64,k-means,NBC3,7.9972,1.2734,4.1408,19.9641,219.2954,0.9498
5,synthetic,3: 8bin no projection quantile NBC3 ABS,67.0906,-1.3362,5433,8098,24,none,quantile,NBC3,7.7621,1.9662,3.9575,8.6304,229.0337,0.9054
6,synthetic,1: 9bin no projection kmeans thermometer SIMD4,66.8396,-1.5873,5412,8097,64,none,k-means,thermometer,5.1379,2.0335,4.0719,10.7435,203.6013,1.0005
7,synthetic,2: 9bin MH PCA64 kmeans NBC4 ABS (9-bin extens...,66.0067,-2.4201,5301,8031,32,PCA64,k-means,NBC4,8.0272,1.0816,4.3443,19.9259,220.3603,0.9444
8,synthetic,3: 9bin no projection quantile NBC4 ABS,67.6957,-0.7311,5482,8098,32,none,quantile,NBC4,6.9900,1.7419,3.8248,8.0125,205.5978,1.0090
9,HPatches,official XFeat cosine MNN (-1),67.2952,0.0000,7138,10607,256,none,None,float32,15.1966,1.0000,7.9995,16.8780,219.8052,1.0000


SECONDARY: extraction-inclusive pipeline, directly measured with interleaved method order.


,dataset,method,pipeline_ms,p10_ms,p90_ms,paired_speedup
0,synthetic,official XFeat cosine MNN (-1),207.3677,197.6231,508.6662,1.0000
1,synthetic,8bin / 7bit AVX2 SIMD4,234.1249,197.8802,397.7593,0.9909
2,synthetic,9bin / 8bit AVX2 SIMD4,241.5989,197.9911,316.8622,0.8774
3,synthetic,1: 8bin no projection kmeans thermometer SIMD4,207.2498,197.7130,355.8308,1.0006
4,synthetic,2: 8bin MH PCA64 kmeans NBC3 ABS (paper PCA-MQ),219.2954,203.6094,416.3595,0.9498
5,synthetic,3: 8bin no projection quantile NBC3 ABS,229.0337,196.2371,380.8741,0.9054
6,synthetic,1: 9bin no projection kmeans thermometer SIMD4,203.6013,197.5280,644.0378,1.0005
7,synthetic,2: 9bin MH PCA64 kmeans NBC4 ABS (9-bin extens...,220.3603,203.0532,538.8202,0.9444
8,synthetic,3: 9bin no projection quantile NBC4 ABS,205.5978,196.3424,412.6654,1.0090
9,HPatches,official XFeat cosine MNN (-1),219.8052,212.3273,380.9733,1.0000


cached_ms is measured separately and may exceed total_ms under runtime noise; inspect raw samples.


## 결과 저장

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo
run_id=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
result_dir=RESULT_ROOT/'fair_comparison_8_9bin_three_methods'/run_id
result_dir.mkdir(parents=True,exist_ok=True)
comparison.to_csv(result_dir/'comparison.csv',index=False)
pd.DataFrame(accuracy_results).to_csv(result_dir/'accuracy.csv',index=False)
pd.DataFrame(pair_accuracy).to_csv(result_dir/'accuracy_per_pair.csv',index=False)
speed_summary.to_csv(result_dir/'speed_summary.csv',index=False)
pipeline_summary.to_csv(result_dir/'pipeline_summary.csv',index=False)
pd.DataFrame(raw_timing).to_csv(result_dir/'raw_timing.csv',index=False)
pd.DataFrame(timing_results).to_csv(result_dir/'timing_per_pair.csv',index=False)
np.savez(result_dir/'fixed_edges.npz',**{f'edges_{b}bin':v.edges for b,v in variants.items()})
parameter_arrays={'mean':mh_mean,'components':mh_components}
for b,pars in mh_parameters.items():
 for k,v in pars.items():
  if isinstance(v,np.ndarray):parameter_arrays[f'{b}bin_{k}']=v
np.savez(result_dir/'manhattan_parameters.npz',**parameter_arrays)
(result_dir/'manhattan_kernel.cpp').write_text(mh_cpp_source)
report=dict(experiment='fair_comparison_8_9bin_three_methods',run_id=run_id,methods=METHODS,
 seed=SEED,timing_seed=TIMING_SEED,top_k=TOP_K,threads=CPU_THREADS,torch_threads=torch.get_num_threads(),
 torch_interop_threads=torch.get_num_interop_threads(),warmup=WARMUP,repeats=REPEATS,pipeline_repeats=PIPELINE_REPEATS,
 platform=platform.platform(),cpu_info=subprocess.run(['lscpu'],capture_output=True,text=True).stdout,
 versions=dict(numpy=np.__version__,torch=torch.__version__,opencv=cv2.__version__),
 accuracy=accuracy_results,pair_accuracy=pair_accuracy,timing=timing_results,pipeline=pipeline_results,
 raw_timing=raw_timing,pair_manifest=pair_manifest,xfeat_provenance=xfeat_provenance,manhattan=mh_provenance,
 official_min_cossim=OFFICIAL_MIN_COSSIM,comparison=comparison.astype(object).where(pd.notna(comparison),None).to_dict('records'),
 note='One runtime, one fixed extractor, shared descriptors/calibration; fixed two threads. Every nine rounds balances all method positions. Primary total includes projection if used, encoding, packing, B layout, and MNN including NBC unpack but excludes extraction. Pipeline includes official extraction twice and output coordinates, excludes image I/O and input tensor preparation. Warmup, calibration and compilation excluded. Speedup is median of paired per-pair median ratios. P10/P90 describe sample spread, not confidence intervals.')
(result_dir/'results.json').write_text(json.dumps(report,indent=2,allow_nan=False))
(result_dir/'official_match_source.py').write_text(official_source)
print('Saved comparison, speed, precision and raw samples:',result_dir)


Saved comparison, speed, precision and raw samples: /content/drive/MyDrive/Colab Notebooks/success-direct-thermometer-AVX2-SIMD4/results/fair_comparison_8_9bin_three_methods/20261008_175643_406834
